# 02 - Text Cleaning & Preprocessing
## Preparing Raw Reviews for NLP & Deep Learning

This notebook handles the cleaning pipeline:
1. **HTML Tag Stripping**: Removing `<br />` tags and leftover markup.
2. **Contraction Expansion**: Expanding contractions (`don't` $	o$ `do not`, `can't` $	o$ `cannot`) to preserve negation tokens.
3. **Whitespace & Character Normalization**: Removing excessive whitespace, normalizing Unicode quotes and hyphens.
4. **Before vs. After Comparison**: Measuring character/token impact.
5. **Cleaned Dataset Export**: Saving to `data/processed/train_reviews_clean.csv`.


In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Load compiled reviews
df = pd.read_csv("../data/processed/train_reviews.csv")
print(f"Loaded {len(df)} reviews.")
df.head(2)


### 1. Defining Text Normalization & Contraction Mapping


In [ ]:
# Common English contractions dictionary
CONTRACTIONS = {
    "ain't": "am not", "aren't": "are not", "can't": "cannot", "can't've": "cannot have",
    "'cause": "because", "could've": "could have", "couldn't": "could not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
    "how'd": "how did", "how'll": "how will", "how's": "how is", "i'd": "I would",
    "i'll": "I will", "i'm": "I am", "i've": "I have", "isn't": "is not", "it'd": "it would",
    "it'll": "it will", "it's": "it is", "let's": "let us", "mightn't": "might not",
    "mustn't": "must not", "shan't": "shall not", "she'd": "she would", "she'll": "she will",
    "she's": "she is", "shouldn't": "should not", "that's": "that is", "there's": "there is",
    "they'd": "they would", "they'll": "they will", "they're": "they are", "they've": "they have",
    "wasn't": "was not", "we'd": "we would", "we'll": "we will", "we're": "we are",
    "we've": "we have", "weren't": "were not", "what'll": "what will", "what're": "what are",
    "what's": "what is", "what've": "what have", "where'd": "where did", "where's": "where is",
    "who'll": "who will", "who's": "who is", "won't": "will not", "wouldn't": "would not",
    "you'd": "you would", "you'll": "you will", "you're": "you are", "you've": "you have"
}

def expand_contractions(text):
    """Expands conversational English contractions into full words."""
    pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in CONTRACTIONS.keys()) + r')\b', flags=re.IGNORECASE)
    def replace(match):
        word = match.group(0).lower()
        expanded = CONTRACTIONS.get(word, match.group(0))
        return expanded
    return pattern.sub(replace, text)

def clean_review_text(text):
    """End-to-end cleaning function."""
    if not isinstance(text, str):
        return ""
    # 1. Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # 2. Expand contractions (preserves critical negation words)
    text = expand_contractions(text)
    # 3. Normalize quotes and dashes
    text = re.sub(r'[‘’]', "'", text)
    text = re.sub(r'[“”]', '"', text)
    # 4. Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


### 2. Testing Preprocessing on Sample Text


In [ ]:
sample_text = df["review"].iloc[0]
print("ORIGINAL:")
print(sample_text[:350])
print("\n" + "="*70 + "\n")
print("CLEANED:")
print(clean_review_text(sample_text)[:350])


### 3. Applying Cleaning to Entire Dataset


In [ ]:
df["clean_review"] = df["review"].apply(clean_review_text)
df["clean_length"] = df["clean_review"].apply(len)

# Verify no nulls or empty reviews were produced
empty_count = (df["clean_length"] == 0).sum()
print(f"Empty reviews after cleaning: {empty_count}")
print(f"Original average length: {df['char_length'].mean():.1f} chars")
print(f"Cleaned average length:  {df['clean_length'].mean():.1f} chars")


### 4. Exporting Cleaned Dataset


In [ ]:
clean_output_path = Path("../data/processed/train_reviews_clean.csv")
cols_to_save = ["review", "label", "sentiment", "clean_review", "clean_length"]
df[cols_to_save].to_csv(clean_output_path, index=False)
print(f"Saved cleaned dataset to: {clean_output_path.resolve()}")
